In [ ]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Coordenadas extraídas de la imagen
        self.start = (0, 0)
        self.walls = {(0, 3), (1, 1), (2, 4), (4, 2)}
        self.slippery_states = {(1, 2), (2, 1), (3, 3)}

        self.terminal_states = {
            (0, 5): 10.0,   # Entrega
            (2, 2): 2.0,    # Estación de carga
            (3, 5): -10.0   # Peligro mortal
        }

        self.danger_states = {
            (1, 4): -3.0,   # Peligro
            (4, 1): -3.0    # Peligro
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if self.is_terminal(state):
            return self.terminal_states[state]
        if state in self.danger_states:
            return -3.0  # El costo de vivir más la penalización del peligro
        return self.living_reward

    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]

        is_slippery = state in self.slippery_states
        p_intended = 0.60 if is_slippery else 0.90
        p_perpendicular = 0.20 if is_slippery else 0.05

        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_intended),
            (perp1, p_perpendicular),
            (perp2, p_perpendicular),
        ]

        transitions = []
        for next_action, prob in outcomes:
            next_state = (
                state[0] + next_action[0],
                state[1] + next_action[1],
            )
            if not self.is_valid_state(next_state):
                next_state = state
            transitions.append((next_state, prob))

        # Agrupar probabilidades si múltiples desvíos llevan al mismo estado (ej. chocar con pared)
        merged_transitions = {}
        for s_prime, p in transitions:
            merged_transitions[s_prime] = merged_transitions.get(s_prime, 0.0) + p

        return list(merged_transitions.items())

# --- ALGORITMOS ---

def expected_next_value(grid, state, action, V):
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )

def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for i in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                best_val = max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )
                V_new[state] = grid.get_reward(state) + grid.gamma * best_val
            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))

        V = V_new
        if biggest_change < threshold:
            return V, i + 1
    return V, max_iter

def extract_policy(grid, V):
    policy = {}
    for state in grid.states():
        if not grid.is_terminal(state):
            policy[state] = max(
                grid.actions,
                key=lambda action: expected_next_value(grid, state, action, V)
            )
    return policy

def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}
    for i in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0
        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                action = policy[state]
                V_new[state] = grid.get_reward(state) + grid.gamma * expected_next_value(grid, state, action, V)
            biggest_change = max(biggest_change, abs(V_new[state] - V[state]))
        V = V_new
        if biggest_change < threshold:
            return V, i + 1
    return V, max_iter

def policy_improvement(grid, V):
    return extract_policy(grid, V)

def policy_iteration(grid, threshold=1e-4, max_iter=100):
    policy = {
        state: grid.actions[0]
        for state in grid.states()
        if not grid.is_terminal(state)
    }
    history = []

    for i in range(max_iter):
        V, eval_iters = policy_evaluation(grid, policy, threshold)
        new_policy = policy_improvement(grid, V)

        changed = sum(new_policy[s] != policy[s] for s in new_policy)
        history.append({
            "policy_iteration": i + 1,
            "evaluation_sweeps": eval_iters,
            "changed_actions": changed,
        })
        policy = new_policy
        if changed == 0:
            break

    V, _ = policy_evaluation(grid, policy, threshold)
    return policy, V, history

### Parte 5 — Interpreta la política

**1. Desde `START`, ¿el robot busca la entrega +10 o prefiere la estación de carga +2?**

Con los parámetros iniciales (`living_reward = -1` y el piso resbaloso), el robot preferirá la estación de carga `+2`. El camino hacia el `+10` es demasiado largo y riesgoso; la acumulación de `-1` por cada paso anula el beneficio del `+10`, haciendo que terminar rápido en el `+2` genere un retorno esperado mayor.

**2. ¿Por qué una recompensa menor podría ser óptima?**

Porque la recompensa mayor está lejos e implica incurrir repetidamente en el costo de vida (`-1`), la atenuación del factor de descuento ($\gamma = 0.9$) y el riesgo de caer en casillas de peligro (`-3` o `-10`). Matemáticamente, el valor presente de `+2` en pocos pasos es superior al de `+10` en muchos pasos.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**

Cambiará la decisión en las celdas adyacentes a las zonas resbalosas (como `(1, 2)`, `(2, 1)`, y `(3, 3)`). El robot intentará abordarlas apuntando hacia una pared (para que un desvío lo estrelle sin moverse en lugar de alejarlo) o buscará rodearlas para evitar perder el control y terminar en las casillas de penalización.

**4. ¿Qué papel cumple el costo por paso `-1`?**

Funciona como una penalización por tiempo o por gasto de batería. Obliga al agente a ser eficiente y buscar el camino más corto hacia un estado terminal, en lugar de deambular por zonas seguras indefinidamente.

**5. ¿Por qué $T(s,a,s')$ ya no puede implementarse con las mismas probabilidades para todos los estados?**

Porque el entorno tiene dinámicas heterogéneas. Las baldosas de piso normal tienen un 90% de éxito en el movimiento intentado, mientras que las de piso resbaloso solo tienen un 60%. La función de transición debe evaluar en qué tipo de celda está parado el robot antes de poder asignar las probabilidades de movimiento.

**Experimentos:**

* **Experimento A (living_reward = -0.1):** Al ser tan barato dar pasos, el robot ya no tiene prisa. Ignorará el `+2` y tomará el camino largo y seguro (rodeando peligros y zonas resbalosas) hacia la entrega de `+10`.

* **Experimento B (Piso muy resbaloso 0.40):** El robot se volverá extremadamente conservador alrededor de los hexágonos amarillos. Usará las paredes de las estanterías de forma deliberada para "frenar" y corregir su rumbo sin riesgo de derrapar hacia los peligros.

* **Experimento C ($\gamma = 0.99$):** Al tener más "paciencia" (el futuro lejano pierde menos valor), la recompensa de `+10` mantendrá gran parte de su magnitud incluso a 7 u 8 pasos de distancia. El robot optará por ir hacia la entrega en lugar de conformarse con el `+2`.